# Drone Chase Tag: PPO eğitimi

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → A100 GPU** seç.
2. Hücreleri sırayla çalıştır. Sonuçlar Google Drive'a yazılır: oturum kapanırsa aynı hücreleri tekrar çalıştır,
   eğitim checkpoint'ten devam eder.
3. Eğitim log'u (`train_log.txt`), değerlendirme (`eval_*.json`) ve grafik (`curves.png`) koşu klasörüne
   otomatik kaydedilir. En sondaki **Arşiv** hücresi bunları zip olarak bilgisayarına indirir.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir("/content")                        # hücre nerede çalışırsa çalışsın, önce sabit bir yere geç
REPO = "https://github.com/HasaNSaygili/DroneChaseTag.git"
if os.path.isdir("/content/proje/.git"):    # daha önce indirildiyse silmeden güncelle
    !git -C /content/proje fetch -q origin && git -C /content/proje reset -q --hard origin/main
else:
    !git clone -q {REPO} /content/proje
assert os.path.isfile("/content/proje/scripts/train.py"), "Kod indirilemedi: bu hücrenin çıktısını sohbete yapıştır"
os.chdir("/content/proje")
!git log --oneline -1
!pip install -q "crazyflow[gpu]==0.4.0" -e .

## Koşu ayarları
Aşağıdaki `RUNS` listesinde bütün koşular ayarlarıyla duruyor; `RUN`'a çalıştırmak istediğinin adını yaz.
Her koşunun tam ayarları ayrıca Drive'da `results/<RUN>/config.json` dosyasına yazılır.

In [ ]:
# Bütün koşular burada kayıtlı (adlar experiments/LOG.md ile aynı). Hiçbiri silinmez, yenisi alta eklenir.
# Her koşu: (script, ayarlar). "train" = tek rol sabit rakibe karşı, "selfplay" = iki rol sırayla.
# Çalıştırmak istediğin koşunun adını RUN'a yaz.
# NOT: 1–6 eylem ortalaması SINIRSIZ eski kodla koştu (doyma kusuru, bkz. LOG); 7'den itibaren ortalama tanh ile sınırlı.
RUNS = {
    # 1) Kovalayan vs OPEN kaçanı, WCT kuralı → reward hacking (kaçanı rezonansla yere düşürdü)
    "ppo-chaser-empty-s0": ("train", ""),
    # 2) Aynısı, rakibi çarptırmak ödülsüz → kovalayan ebelemeyi öğrendi (%99,8)
    "ppo-chaser-empty-nocrashrew-s0": ("train", "opp_crash_reward=0.0"),
    # 3) Kaçan vs elle yazılmış kovalayan; γ 0.998 (kaçanın ödülü uzakta) → kaçan %99,8 dayandı
    "ppo-evader-empty-nocrashrew-s0": ("train", "learner=evader opponent=scripted_chaser opp_crash_reward=0.0 gamma=0.998"),
    # 4) Self-play, boş arena, WCT kuralı. Havuz 2 ve 3 numaralı koşuların politikalarıyla başlar. 20 nesil.
    #    → kovalayan baskın (kaçan en son rakibe karşı %0), kaçanın std'si 0,03'e çöktü
    "sp-empty-s0": ("selfplay", ""),
    # 5) Kontrol: 4'ün aynısı + entropi bonusu → denge değişti: kaçan %15–61, taraflar sırayla birbirini yeniyor
    "sp-empty-ent-s0": ("selfplay", "ent_coef=0.003"),
    # 6) Faz 1 engeller, 8–12 (~%11 dolu) + entropi → BAŞARISIZ: doymuş eylemler, drone'lar engellere çarpıp durdu
    "sp-faz1-s0": ("selfplay", '"n_obst=(8,12)" ent_coef=0.003'),
    # 7) Doğrulama, engelli arena, SIFIRDAN: kovalayan vs OPEN kaçanı, rakibi çarptırmak ödülsüz
    "ppo-chaser-faz1-s0": ("train", '"n_obst=(8,12)" opp_crash_reward=0.0 ent_coef=0.003'),
    # 8) Doğrulama, engelli arena, SIFIRDAN: kaçan vs elle yazılmış kovalayan (yavaş rakip)
    "ppo-evader-faz1-s0": ("train", 'learner=evader opponent=scripted_chaser "n_obst=(8,12)" opp_crash_reward=0.0 gamma=0.998 ent_coef=0.003'),
    # 9) Self-play, boş arena, WCT kuralı, DÜZELTİLMİŞ kod (tanh) + ent 0.001. Havuz sweep-base-empty'nin
    #    politikalarıyla başlar (kovalayan %99,6, kaçan %100). → Dengeli (kaçan ~%50), ama sıra değişimi ölçüsü YANLIydı
    #    (senkron başlangıç: süre dolan maçlar aynı turda bitip kaçanın oranını şişirdi; 4–6'da da vardı).
    "sp-empty-fix-s0": ("selfplay", "ent_coef=0.001 init_chaser=experiments/ppo-chaser-empty-fix-s0/params_00476.pkl "
                                   "init_evader=experiments/ppo-evader-empty-fix-s0/params_00476.pkl"),
    # 10) 9'un aynısı, sıra değişimi AYRI KONTROL MAÇLARIYLA (4096 maç baştan sona, her 5 M adımda bir).
    "sp-empty-fix2-s0": ("selfplay", "ent_coef=0.001 init_chaser=experiments/ppo-chaser-empty-fix-s0/params_00476.pkl "
                                    "init_evader=experiments/ppo-evader-empty-fix-s0/params_00476.pkl"),
    # 11) Self-play, ENGELLİ arena (faz 1, 8–12 engel), WCT kuralı, kontrol maçlı: 10'un aynısı, sadece arena farklı
    #     (ablasyon 3'ün engelli tarafı). Havuz sweep-base-faz1'in politikalarıyla başlar: kovalayan
    #     ppo-chaser-faz1-long (OPEN'a %82), kaçan ppo-evader-faz1-long-s0 (elle yazılmış kovalayana %61).
    #     Soru: engeller yeni taktik getiriyor mu? (temel kaçanlar engelleri kullanmıyor, daire çiziyordu)
    "sp-faz1-fix2-s0": ("selfplay", '"n_obst=(8,12)" ent_coef=0.001 '
                                   "init_chaser=experiments/ppo-chaser-faz1-long/params_01430.pkl "
                                   "init_evader=experiments/ppo-evader-faz1-long-s0/params_01430.pkl"),
    #     → nesil 8'de kovalayan ÇÖKTÜ (kendini tavana çarpmayı öğrendi): aşırı büyük PPO güncellemeleri
    #       (clip_frac 0,4, kl 0,05). Düzeltme: KL ile erken durdurma + öğrenme oranı ısınması (12 ve 13).
    # 12) 11'in aynısı + target_kl=0.02 (bir turda politika bu kadar değişince turun kalanı atlanır) +
    #     warmup_steps=100 (taze Adam'ın ilk adımları lr 0'dan başlar; kaçanın ilk turunda kl 3,1'di).
    "sp-faz1-kl-s0": ("selfplay", '"n_obst=(8,12)" ent_coef=0.001 target_kl=0.02 warmup_steps=100 '
                                 "init_chaser=experiments/ppo-chaser-faz1-long/params_01430.pkl "
                                 "init_evader=experiments/ppo-evader-faz1-long-s0/params_01430.pkl"),
    # 13) Boş arena karşılığı (10'un aynısı + aynı iki düzeltme): ablasyon 3'te 12 ile aynı kodla kıyas için.
    "sp-empty-kl-s0": ("selfplay", "ent_coef=0.001 target_kl=0.02 warmup_steps=100 "
                                  "init_chaser=experiments/ppo-chaser-empty-fix-s0/params_00476.pkl "
                                  "init_evader=experiments/ppo-evader-empty-fix-s0/params_00476.pkl"),
    #     → 12 ve 13'te fren %80–97 devrede, işleri KÖTÜLEŞTİRDİ: her turun ilk adımı Adam'ın momentumuyla zaten büyük
    #       (kl > 0,02), fren kalan 31 düzeltici adımı atlıyor, sadece momentumun itişi kalıyor. CPU tanısı (2048 dünya):
    #       tur başına ort. kl frensiz 0,010–0,031, frenli 0,031–0,098. Fren kaldırıldı; adım boyu lr ile küçültülür.
    # 14) 11'in aynısı + lr=1e-4 (3e-4 yerine; kl ~ lr², düşük std'li politikalar hassas) + warmup_steps=100, fren YOK.
    #     CPU tanısı: ort. kl 0,0045–0,0063, ilk adım ≤ 0,005, clip %5–8, sıçrama yok.
    "sp-faz1-lr1e4-s0": ("selfplay", '"n_obst=(8,12)" ent_coef=0.001 lr=0.0001 warmup_steps=100 '
                                    "init_chaser=experiments/ppo-chaser-faz1-long/params_01430.pkl "
                                    "init_evader=experiments/ppo-evader-faz1-long-s0/params_01430.pkl"),
    # 15) Boş arena karşılığı (10'un aynısı + aynı ayarlar): ablasyon 3'te 14 ile AYNI ayarla kıyas için.
    "sp-empty-lr1e4-s0": ("selfplay", "ent_coef=0.001 lr=0.0001 warmup_steps=100 "
                                     "init_chaser=experiments/ppo-chaser-empty-fix-s0/params_00476.pkl "
                                     "init_evader=experiments/ppo-evader-empty-fix-s0/params_00476.pkl"),
    #     → 14 ve 15 kararlı (çöküş yok). Boş arena en temiz self-play (Elo: kovalayan +587, kaçan +900). Engelli:
    #       kovalayan baskın (Elo +256, 4. sürümden sonra düz; kaçan +95), kaçanın std'si 0,059 → 0,028'e çöktü.
    # 16) 14'ün aynısı + evader_init_std=0.2: self-play başında öğrenen kaçanın std'si 0,064 → 0,2 (boş arena kaçanı
    #     seviyesi; ağ aynı, havuzdaki kaçan_0 değişmez). Soru: kovalayan baskınlığı keşif çöküşünden mi?
    "sp-faz1-evstd-s0": ("selfplay", '"n_obst=(8,12)" ent_coef=0.001 lr=0.0001 warmup_steps=100 evader_init_std=0.2 '
                                    "init_chaser=experiments/ppo-chaser-faz1-long/params_01430.pkl "
                                    "init_evader=experiments/ppo-evader-faz1-long-s0/params_01430.pkl"),
    # 17) 14'ün aynısı + ent_coef_evader=0.003: sadece kaçanın entropi bonusu 3 katı (kovalayan 0.001'de kalır).
    #     16'dan farkı: gürültüyü bir kerede değil, eğitim boyunca yukarı iter (lr 1e-4 ile yavaş).
    "sp-faz1-event-s0": ("selfplay", '"n_obst=(8,12)" ent_coef=0.001 lr=0.0001 warmup_steps=100 ent_coef_evader=0.003 '
                                    "init_chaser=experiments/ppo-chaser-faz1-long/params_01430.pkl "
                                    "init_evader=experiments/ppo-evader-faz1-long-s0/params_01430.pkl"),
}
RUN = "sp-faz1-evstd-s0"
SCRIPT, SET = RUNS[RUN]
OUT = "/content/drive/MyDrive/DroneChaseTag/results"
print(RUN, "|", SCRIPT, "|", SET or "(varsayılan ayarlar)")

In [ ]:
import os
os.chdir("/content/proje")   # kurulum hücresi çalışmadıysa burada hata verir
!python -W ignore scripts/{SCRIPT}.py --name {RUN} --out {OUT} {('--set ' + SET) if SET else ''} 2>&1 | grep -v -E "warp|spec|fromstring"

## Değerlendirme ve video
Hiç görülmemiş arenalarda, gürültüsüz oynatır; 3 maçı videoya çeker (Drive'da `videos/`).

In [ ]:
import os
os.chdir("/content/proje")
if SCRIPT == "train":
    !MUJOCO_GL=egl python -W ignore scripts/watch_policy.py --run {OUT}/{RUN} --games 3 --eval-worlds 1024 --out /content/drive/MyDrive/DroneChaseTag/videos 2>&1 | grep -v -E "warp|spec|fromstring"
else:
    print("Self-play değerlendirmesi (Elo turnuvası ve videolar) ayrı bir adımda yapılacak.")

## Eğitim eğrisi

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open(f"{OUT}/{RUN}/metrics.jsonl")]
steps = np.array([r.get('steps', r.get('total_steps')) for r in rows]) / 1e6
C = np.array([r['outcome_counts'] for r in rows]); tot = C[:, 1:].sum(1, keepdims=True).clip(1)
names = ['ebeleme', 'süre doldu', 'kaçan çarptı', 'kovalayan çarptı']
K = 5   # kayan ortalama penceresi; 'valid': kenarlarda sahte düşüş olmasın diye uçlar atılır
smooth = lambda y: np.convolve(y, np.ones(K) / K, 'valid')
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for i, n in enumerate(names):
    ax[0].plot(steps[K // 2: len(steps) - K // 2], smooth(C[:, i + 1] / tot[:, 0]), label=n)
ax[0].set_title('sonuç oranları (eğitim maçları)'); ax[0].set_xlabel('M adım'); ax[0].legend()
ax[1].plot(steps, [r['ep_len'] / 50 for r in rows]); ax[1].set_title('ortalama maç süresi (s)')
if 'gen' in rows[0]:   # self-play: öğrenenin en son rakibe karşı kazanması; nesil sınırları kesik çizgi
    if 'win_latest_eval' in rows[0]:   # kontrol maçları (her maç tam bir kez sayılır): sadece yapıldığı turlarda
        ev = [(s, r['win_latest_eval']) for s, r in zip(steps, rows) if r['win_latest_eval'] is not None]
        ax[2].plot(*zip(*ev), 'o-', ms=3, label='kontrol maçı: en son rakibe karşı kazanma')
    else:                              # eski koşular: eğitim maçlarından, senkron başlangıç yüzünden yanlı
        ax[2].plot(steps, [r['win_latest_window'] for r in rows], label='en son rakibe karşı kazanma (yanlı ölçü)')
    ax[2].axhline(0.6, color='red', lw=0.6, ls=':')   # sıra değişimi eşiği
    for i in range(1, len(rows)):
        if rows[i]['gen'] != rows[i - 1]['gen']:
            for a in ax:
                a.axvline(steps[i], color='gray', lw=0.6, ls='--')
    ax[2].set_ylim(0, 1); ax[2].legend(); ax[2].set_title('self-play (nesiller kesik çizgiyle)')
else:
    ax[2].plot(steps, [r['explained_var'] for r in rows], label='explained var')
    ax[2].plot(steps, [r['std'] for r in rows], label='std'); ax[2].legend(); ax[2].set_title('critic ve keşif')
fig.suptitle(RUN)
fig.savefig(f"{OUT}/{RUN}/curves.png", dpi=120, bbox_inches="tight")   # koşu klasörüne kaydet
plt.show()

## Arşiv: koşunun dosyalarını bilgisayarına indir
config, log, ölçümler, değerlendirme, grafik ve son politika (checkpoint ve ara politikalar hariç: büyük)
tek bir zip olarak iner. Zip'i sohbette söyle; `experiments/<RUN>/` altına konup repoya eklenecek.
Başka bir koşuyu arşivlemek için aşağıda `ARCHIVE = "<koşu adı>"` yaz (Drive'da duran eski koşular için de çalışır).

In [ ]:
import glob, os, zipfile
from google.colab import files
ARCHIVE = RUN                      # eski bir koşu için: ARCHIVE = "ppo-chaser-empty-s0"
d = f"{OUT}/{ARCHIVE}"
keep = [p for p in glob.glob(f"{d}/*") if not p.endswith("checkpoint.pkl") and not os.path.basename(p).startswith("params_")]
params = sorted(glob.glob(f"{d}/params_*.pkl"))
keep += params[-1:]                # sadece son politika (makaledeki sayıların geldiği)
zpath = f"/content/{ARCHIVE}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p in keep:
        z.write(p, f"{ARCHIVE}/{os.path.basename(p)}")
print("\n".join(os.path.basename(p) for p in keep))
files.download(zpath)